# 04b · Vazamento de casos: hold-out de Lira et al. × dobras agrupadas (mesma receita)

**Sessão P1 · ambiente L4.** Isola o efeito da **partição** sobre os números de Lira et al. (2025, IJLM): mesma receita, mesmo código e mesmas cabeças, mudando só como treino e teste são separados.

* **Partição de Lira (com vazamento):** reprodução fiel de `iterative_split` (`leguwoi/src/data/gunshot_dataset.py`): `IterativeStratification` (scikit-multilearn 0.2.0) sobre o one-hot de `[case_number, alvo]`, 20 % de teste, aplicado aos CSVs do próprio dataset — `gunshots_eqx.csv` com `shot_type` (entrada/saída) e `gunshots_eqx_ONLY_INPUT_SHOTS.csv` com `distance` (MLSD). Lira não fixou semente (o desempate usa o RNG global do NumPy); aqui há **3 partições** (`np.random.seed` 42, 43, 44): reproduz-se o **método**, não a partição exata do artigo. O número do caso sem o ano mistura casos de anos diferentes, como no original.
* **O que se mede:** a fração das imagens de teste cujo caso (ano.caso) também está no treino, e as métricas da ResNet-152 com cabeça **linear (a)** e **KAN (d)** nas duas tarefas — 12 unidades (~1,5 h na L4). A comparação é com as 5 dobras agrupadas (cada uma também é um *hold-out* ~80/20) da semente 42 do 02/04.
* **Única diferença de código:** o protocolo recusa caso em treino e teste; para reproduzir o split de Lira, a unidade de agrupamento passa a ser a **imagem** (`case_id = rel_path`, ver `cm.sessao_holdout`). Isso fica registrado em `dados` de cada `metricas.json`.
* `dados/holdout_lira2024.csv` guarda só índices (0 teste, 1 treino, −1 fora do CSV de Lira) e é gravado uma única vez.

**Como rodar:** `TESTE_RAPIDO = True` na 1ª vez; depois `False` e **Executar tudo** (retoma sozinho se cair).

In [ ]:
# ==== Configuração desta sessão ====
DIR_DRIVE = "/content/drive/MyDrive/Colab Notebooks/tcc-final"  # pasta tcc-final (conta padrão, authuser=0)
RAIZ_TRABALHO = "/content"   # onde o dataset é clonado e o cache é gerado (disco local, temporário)
SHA_PROTOCOLO_ESPERADO = "b7cd9baca53edd96c855cca10914fc258efa8ddfd0937cc0d3e9572de5ef9e13"
SHA_PLUGIN_ESPERADO = "e2138640a83ef7587200ad6e051262b6a49e5b60027f6b4a61407fd1f61889f7"   # configs_mlsd.py
SHA_HOLDOUT_LOCAL = "4391f9c4ece7d26510222717e987ffb054da5fb9673ef522071e268b9574e533"   # informativo: o CSV que o P1 gerou localmente
ABA = "B"                    # dono das travas desta aba
PERMITIR_RESERVA = False     # True só para rodar na T4 como reserva (marcado com † na tabela)
SO_PARTICOES = None          # ex.: [42] para rodar só uma partição
TESTE_RAPIDO = True          # 1ª vez True; depois False
CKPT_INTERVALO_MIN = None    # None = usa a recomendação do 01

In [ ]:
# Dependências fixadas (rode ANTES de importar qualquer coisa da albumentations)
!pip install -q "albumentations==2.0.8" "git+https://github.com/Blealtan/efficient-kan@7b6ce1c87f18c8bc90c208f6b494042344216b11" "scikit-multilearn==0.2.0"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import sys, os, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, DIR_DRIVE)
import protocolo_comum as pc

pc.checar_sentinela_drive(DIR_DRIVE)   # PARA aqui se o Colab estiver logado em outra conta
assert pc.sha256_protocolo() == SHA_PROTOCOLO_ESPERADO, (
    "O protocolo_comum.py do Drive não é o publicado. Não rode nada; avise o P1.")
ARQ_PLUGIN = f"{DIR_DRIVE}/configs_mlsd.py"
assert pc.sha256_arquivo(ARQ_PLUGIN) == SHA_PLUGIN_ESPERADO, (
    "O configs_mlsd.py do Drive não é o publicado. Não rode nada; avise o P1.")
pc.carregar_plugin(ARQ_PLUGIN)          # registra as configs mlsd__* (o SHA vai para cada metricas.json)
cm = sys.modules["configs_mlsd"]
print("Dependências:", pc.checar_dependencias())
print("Protocolo", pc.PROTOCOLO_VERSAO, "| SHA-256", pc.sha256_protocolo()[:16], "… | plugin", SHA_PLUGIN_ESPERADO[:16], "…")
GPU = pc.exigir_gpu(permitir_reserva=PERMITIR_RESERVA)   # rodadas comparáveis só na L4
pc.definir_id_sessao(ABA)
print("GPU:", GPU["nome"], "| comparável:", GPU["comparavel"], "| dono das travas:", pc.ID_SESSAO)

In [ ]:
# Clona o dataset e o leguwoi nos commits fixados e gera o cache 224×224 só nesta sessão (~3–5 min)
sess = pc.preparar_sessao(
    RAIZ_TRABALHO,
    caminho_dobras=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_DOBRAS}",
    caminho_meta=f"{DIR_DRIVE}/dados/{pc.ARQUIVO_META_DOBRAS}",
)
R, C = f"{DIR_DRIVE}/resultados", f"{DIR_DRIVE}/checkpoints"
RT, CT = f"{DIR_DRIVE}/resultados_teste", f"{DIR_DRIVE}/checkpoints_teste"
print(json.dumps(sess.meta, indent=1))
if sess.meta.get("cache_confere") is False:
    print("⚠️ O cache diverge do registrado no 00 (versão do OpenCV?). Divergência:", sess.meta.get("cache_divergencia"))

## Partições no estilo de Lira (gravadas uma única vez) e taxa de vazamento

In [ ]:
listagem = pc.listar_imagens(sess.raiz_dataset)            # as 2.554 imagens (independe de subamostras)
holdout = cm.tabela_holdout_lira(sess.raiz_dataset, listagem)
arq = Path(DIR_DRIVE) / "dados" / "holdout_lira2024.csv"
novo = holdout.to_csv(index=False, lineterminator="\n")
if arq.exists():
    if arq.read_text(encoding="utf-8") != novo:
        raise RuntimeError("Já existe um holdout_lira2024.csv DIFERENTE no Drive (outra versão do scikit-multilearn/NumPy?). "
                           "Não sobrescrevo; avise o P1.")
    print("holdout_lira2024.csv já existe e é idêntico — nada a fazer.")
else:
    tmp = arq.with_name(arq.name + ".tmp")
    tmp.write_text(novo, encoding="utf-8")
    os.replace(tmp, arq)
    print("Gravado:", arq)
SHA_HOLDOUT = pc.sha256_arquivo(arq)
print("SHA-256:", SHA_HOLDOUT, "| igual ao gerado localmente pelo P1:", SHA_HOLDOUT == SHA_HOLDOUT_LOCAL)
resumo_part = cm.resumo_holdout(listagem, holdout)
display(resumo_part)
assert (resumo_part["rotulo_invalido_no_uso"] == 0).all()
arq_meta = Path(DIR_DRIVE) / "dados" / "holdout_lira2024_meta.json"
if not arq_meta.exists():
    import skmultilearn, sklearn
    pc.gravar_json_atomico({
        "gerado_em": pc.agora_iso(), "holdout_sha256": SHA_HOLDOUT, "particoes": list(cm.SEMENTES_PARTICAO),
        "metodo": "leguwoi iterative_split: IterativeStratification(n_splits=2, order=2, [0.2, 0.8]) sobre one-hot "
                  "[case_number, alvo]; np.random.seed(p) antes de cada partição",
        "csvs": cm.CSVS_LIRA, "resumo": resumo_part.to_dict("records"),
        "versoes": {"scikit_multilearn": cm.VERSAO_SKMULTILEARN, "sklearn": sklearn.__version__,
                    "numpy": np.__version__, "pandas": pd.__version__},
        "ambiente": pc.info_ambiente(),
    }, arq_meta)
    print("Gravado:", arq_meta)

In [ ]:
cal = json.loads((Path(R) / "calibracao_l4.json").read_text())
PREC = cal.get("decisao_precisao")
assert PREC in ("amp", "fp32"), "Rode o 01 (ponte fp32 × AMP) antes."
CKPT = cal.get("recomendacao_ckpt_intervalo_min", 0.0) if CKPT_INTERVALO_MIN is None else CKPT_INTERVALO_MIN
sess_v = cm.sessao_holdout(sess, holdout, SHA_HOLDOUT)
E1, H = pc.CONFIGS_EXP1, cm.CONFIGS_CABECA
particoes = [p for p in cm.SEMENTES_PARTICAO if SO_PARTICOES is None or p in SO_PARTICOES]
fila = cm.fila_vazamento({"es": [E1["a"], E1["d"]], "dist": [H["a"], H["d"]]}, particoes=particoes)
print(f"Precisão: {PREC} | checkpoint a cada ≥ {CKPT} min | {len(fila)} unidades:")
display(pd.DataFrame(fila)[["experimento", "config", "semente", "coluna_dobra"]])

## Teste rápido (1 unidade × 2 épocas, pasta de teste)

In [ ]:
if TESTE_RAPIDO:
    import shutil
    for u0 in (fila[0], fila[2]):   # uma de entrada/saída e uma de distância
        u = {**u0, "experimento": "t04b_rapido"}
        shutil.rmtree(f"{RT}/t04b_rapido/{u['config']}", ignore_errors=True)
        shutil.rmtree(f"{CT}/t04b_rapido/{u['config']}", ignore_errors=True)
        r = pc.executar_unidade(sess_v, u, RT, CT, precisao=PREC, limite_epocas=(1, 1), espera_trava_s=0)
        t = next(iter(r["metricas"]))
        print(u["config"], u["coluna_dobra"], "OK:", {k: round(r["metricas"][t][k], 4) for k in ("acc", "auc")},
              f"| treino {r['n_treino']} | teste {r['n_teste']} | tempos", r["tempos"])
    print("➡️ Agora mude TESTE_RAPIDO = False e rode tudo de novo.")

## Rodada

In [ ]:
assert not TESTE_RAPIDO, "TESTE_RAPIDO está True: mude para False para rodar."
feitos = pc.executar_fila(sess_v, fila, R, C, precisao=PREC, ckpt_intervalo_min=CKPT)
print("Concluídas nesta sessão:", len(feitos))

## Resultados (a comparação com as dobras agrupadas está no 05b)

In [ ]:
partes = [pc.ler_resultados(R, f"vazamento_lira_p{p}") for p in cm.SEMENTES_PARTICAO]
runs_v = pd.concat([p for p in partes if not p.empty], ignore_index=True) if any(not p.empty for p in partes) else pd.DataFrame()
if runs_v.empty:
    print("Nada concluído ainda.")
else:
    runs_v = runs_v[~runs_v["teste_rapido"].astype(bool)]
    cols = [c for c in ("es_acc", "es_auc", "es_f1_macro", "dist_acc", "dist_auc", "dist_f1_macro", "dist_acc_balanceada") if c in runs_v]
    runs_v.to_csv(Path(R) / "vazamento_lira_runs.csv", index=False)
    display(runs_v[["experimento", "config", "n_treino", "n_teste", "treino_min"] + cols].round(4))

⚠️ Ao terminar, desconecte o ambiente (*Ambiente de execução → Desconectar e excluir*) para não consumir unidades parado.